# Eksperimen kemiripan/pengulangan audio antarsection

Buka dari root **SongMap**. Kita memakai label manual dan split yang sudah ada, cache section asli, serta konteks epoch 5 yang dibekukan. Tidak perlu anotasi ulang atau menjalankan MuQ/MusicFM.

Dua adapter kecil dilatih dengan pengaturan yang sama: **context_adapter** tanpa fitur pengulangan tambahan dan **repetition_adapter** dengan fitur kemiripan audio antarsection yang tidak bersebelahan. Model konteks lama sudah melihat konteks global; pembanding ini mengukur manfaat tambahan dari fitur pengulangan yang eksplisit.

Setiap adapter sekitar **5.096 parameter**, dilatih di CPU. Teacher kecil berjalan sekali pada perangkat training semula untuk mempertahankan pembanding yang sama. Seluruh target train/val harus mereproduksi audit terakhir sebelum training penuh dimulai. Boundary dan label manual tidak diubah. Ini belum memperbaiki model di aplikasi.


In [ ]:
from pathlib import Path
from datetime import datetime
import json, subprocess

ROOT=Path.cwd().resolve()
assert (ROOT/'scripts/train_repetition_context.py').is_file(), 'Buka notebook dari root SongMap'
SONGSTRUCT_ROOT=ROOT.parent/'Section-Annotations'
RUN_PYTHON=SONGSTRUCT_ROOT/'.venv-songformer311/bin/python'  # Jangan resolve symlink executable venv.
assert RUN_PYTHON.is_file(), f'Python tidak ditemukan: {RUN_PYTHON}'
CACHE=ROOT/'outputs.local/section-context-cache-v1'
assert CACHE.is_dir(), 'Cache section training sebelumnya belum ditemukan; periksa CACHE'
receipt=ROOT/'SongMap/Resources/MLModels/SongFormerContext/songformer_section_context_manifest.json'
EXPECTED_CONTEXT_SHA=(json.loads(receipt.read_text())['checkpointSHA256'] if receipt.is_file()
    else '1511a8c4c53b86a3c58508a4a7c2f890daa1adb926acbdb9dae56ad529d01e01')
candidates=[]
for path in (ROOT/'outputs.local').glob('frozen-context-test-*/selection.json'):
    lock=json.loads(path.read_text())
    if lock.get('context_epoch')==5 and lock.get('context_checkpoint_sha256')==EXPECTED_CONTEXT_SHA and (path.parent/'frozen-context.pt').is_file():
        candidates.append(path)
assert candidates, 'Snapshot context epoch 5 yang cocok belum ditemukan'
FROZEN=max(candidates,key=lambda p:p.stat().st_mtime).parent
candidates=[]
for path in (ROOT/'outputs.local').glob('manual-context-audit-*/audit-all.json'):
    report=json.loads(path.read_text())
    if (report.get('full_train_val') and report.get('original_context_validation_reproduced')
            and report.get('context_checkpoint_sha256')==EXPECTED_CONTEXT_SHA):
        candidates.append(path)
assert candidates, 'Audit manual lengkap yang sudah berhasil belum ditemukan'
AUDIT=max(candidates,key=lambda p:p.stat().st_mtime)
RESUME_RUN=None  # Setelah restart kernel, isi path folder repetition-context-* untuk melanjutkan run lama.
OUTPUT=(Path(RESUME_RUN).expanduser().resolve() if RESUME_RUN else
        ROOT/'outputs.local'/('repetition-context-'+datetime.now().strftime('%Y%m%d-%H%M%S-%f')))
subprocess.run([str(RUN_PYTHON),'-c','import sys,numpy,torch; print("Python:",sys.executable); print("Torch:",torch.__version__)'],check=True)
print('Teacher:',FROZEN)
print('Audit:',AUDIT)
print('Output:',OUTPUT)

def run_experiment(smoke=False):
    command=[str(RUN_PYTHON),'-u',str(ROOT/'scripts/train_repetition_context.py'),
        '--songstruct-root',str(SONGSTRUCT_ROOT),'--frozen-evaluation',str(FROZEN),
        '--cache-dir',str(CACHE),'--audit',str(AUDIT),'--output-dir',str(OUTPUT)]
    if smoke:
        command+=['--smoke-test']
    elif (OUTPUT/'run.json').is_file():
        command+=['--resume']
        print('Melanjutkan:',OUTPUT)
    log=Path(str(OUTPUT)+('.smoke.log' if smoke else '.training.log'))
    log.parent.mkdir(parents=True,exist_ok=True)
    with log.open('a',encoding='utf-8') as handle:
        process=subprocess.Popen(command,cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='');handle.write(line);handle.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    if status:raise RuntimeError(f'Gagal ({status}); lihat {log}. Epoch lengkap yang tersimpan dapat dilanjutkan.')


In [ ]:
# Uji backward nyata pada satu train + satu validation untuk kedua adapter.
# Tidak menyimpan checkpoint atau membuat folder run.
run_experiment(smoke=True)


Lanjutkan setelah dua baris **Smoke OK**. Training penuh membandingkan hasil awal pada seluruh train/val dengan audit sebelumnya, lalu melatih kedua adapter secara terpisah. Early stopping memakai macro F1 validation. Kandidat juga harus mempertahankan F1 bridge dan membatasi penurunan F1 chorus ≤0,01 serta inst ≤0,02 dari model konteks lama. Ini batas eksperimen yang ditetapkan sebelum training, bukan bukti kualitas final.

Jika tidak ada peningkatan yang memenuhi syarat, model lama dipertahankan. Jika pembanding tanpa pengulangan sama baiknya, pembanding yang lebih sederhana dipilih. Hasil test tidak dipakai.


In [ ]:
run_experiment()


In [ ]:
from IPython.display import display, Markdown
report=json.loads((OUTPUT/'comparison.json').read_text())
assert report['full_validation'] and not report['test_used']
lines=['| Model | Epoch | Train acc | Val acc | Train F1 | Val F1 |',
       '|---|---|---|---|---|---|']
b=report['baseline']
lines.append(f"| Context lama | 5 (beku) | {b['train']['section_accuracy']:.2%} | {b['val']['section_accuracy']:.2%} | {b['train']['macro_f1_present']:.4f} | {b['val']['macro_f1_present']:.4f} |")
for mode,r in report['candidates'].items():
    lines.append(f"| {mode} | {r['best_epoch']} | {r['train']['section_accuracy']:.2%} | {r['validation']['section_accuracy']:.2%} | {r['train']['macro_f1_present']:.4f} | {r['validation']['macro_f1_present']:.4f} |")
display(Markdown('\n'.join(lines)))
print('Terpilih:',report['selected'])
print('Delta F1 repetition vs pembanding:',report['repetition_minus_ablation_macro_f1'])
for mode,r in report['candidates'].items():
    print('\n',mode,'guards:',r['guards'],'| model lama dipertahankan:',r['identity_retained'])
    print('Val per-class F1:',{label:round(v['f1'],4) for label,v in r['validation']['per_class'].items()})
    matrix=r['validation']['confusion_matrix'];order=r['validation']['label_order']
    print('Bridge → chorus:',matrix[order.index('bridge')][order.index('chorus')],
          '| Chorus → bridge:',matrix[order.index('chorus')][order.index('bridge')])
print('\nKirim file:',OUTPUT/'comparison.json')


Checkpoint adapter tersimpan terpisah dalam folder masing-masing mode. Model konteks lama, encoder, cache, anotasi, serta boundary tetap utuh. Hasil ini memakai **cache encoder asli**, sehingga belum menilai pipeline palette6/Core ML, waktu iPhone atau akurasi boundary. Validation telah dipakai pada eksperimen sebelumnya; peningkatan di sini perlu dibaca sebagai hasil pemilihan eksperimen, bukan estimasi final yang independen.

Untuk melanjutkan setelah menghentikan proses, jalankan kembali cell training pada kernel yang sama. Setelah restart kernel, isi `RESUME_RUN` dengan folder output sebelumnya lalu jalankan setup dan training. Resume menyimpan optimizer dan RNG pada akhir epoch; epoch yang belum selesai diulang. Perubahan sumber/config/data membuat resume ditolak agar eksperimen tidak tercampur.
